In [ ]:
#Loading target model:
model_path = "" #finetuned target model's path
tokenizer_path = ''

#Load anonymized forget dataset:
forget_data_path = ""

#Load anonymized retain dataset:
retain_data_path = ''

#MLP training:
mlp_path = '' #path to save trained mlp model


#Forget space creation:
forget_space_path = "" #path to save forget space

#Subspace creation:
OUTPUT_SUBSPACE_FILE = "" #path to save forget subspace

#Unlearning filter:
UNLEARNING_FILTER_PATH = "" #path to save unlearning filter

#Active unlearning -- Inference:
LAYER_TO_MODIFY = # 
UNLEARNING_STRENGTH_ALPHA = #

#Save unlearned model:
SAVE_DIRECTORY = ''

In [ ]:
# Import dependencies
from datasets import load_dataset
from random import randrange
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import datasets
from datasets import load_dataset, load_from_disk
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from tqdm import tqdm
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os

In [ ]:
# Load model and datasets
model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",                  
        torch_dtype=torch.bfloat16
    )
model = torch.compile(model)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

forget_dataset = load_from_disk(forget_data_path)
retain_dataset = load_from_disk(retain_data_path)

In [ ]:
# MLP training

# Public datasets, which can accordingly be changed as per usecase
DATASETS_TO_LOAD = {
    "finance": "modeldev/synthetic_pii_finance",
    "health": "Parth/piiforprivacy",
    "customer_interaction": "soates/australian-insurance-pii-dataset-corrected",
    "syn_profile": "ponoma16/implicit_pii_detection",
    "ninja_masker": "King-Harry/NinjaMasker-PII-Redaction",
    "general_pii": "ai4privacy/pii-masking-400k"
}

POSSIBLE_ORIGINAL_COLS = ["Filled Template", "source_text", "original", "unmasked_text", "original_text"]
POSSIBLE_ANONYMIZED_COLS = ["Template", "masked_text", "anonymized", "target_text", "redacted_text"]
MAX_SAMPLES_PER_DATASET = 5000 
TEST_SET_SIZE = 0.15
VALIDATION_SET_SIZE = 0.15

LEARNING_RATE = 1e-4
EPOCHS = 10
BATCH_SIZE = 32

def find_column_name(available_cols, possible_cols):
    for col in possible_cols:
        if col in available_cols:
            return col
    return None

def load_and_prepare_data(datasets_dict, max_samples):
    all_pairs = []
    print("Loading and preparing datasets...")
    for name, path in datasets_dict.items():
        try:
            print(f"-> Loading '{name}' dataset from '{path}'...")
            dataset = load_dataset(path, split='train')
            dataset = dataset.shuffle(seed=42).select(range(min(len(dataset), max_samples)))

            if name == "ninja_masker":
                for record in dataset:
                    text_content = record.get('text', '')
                    if "### Assistant:" in text_content and "### Human:" in text_content:
                        try:
                            parts = text_content.split("### Assistant:")
                            human_part = parts[0].replace("### Human:", "").strip()
                            assistant_part = parts[1].strip()
                            all_pairs.append({
                                "original": human_part,
                                "anonymized": assistant_part
                            })
                        except IndexError:
                            continue
            else:
                original_col = find_column_name(dataset.column_names, POSSIBLE_ORIGINAL_COLS)
                anonymized_col = find_column_name(dataset.column_names, POSSIBLE_ANONYMIZED_COLS)

                if not original_col or not anonymized_col:
                    print(f"  [Warning] Dataset '{name}' is missing required columns. Skipping.")
                    continue
                
                print(f"  -> Found columns: Original='{original_col}', Anonymized='{anonymized_col}'")
                for record in dataset:
                    all_pairs.append({
                        "original": record[original_col],
                        "anonymized": record[anonymized_col]
                    })
            
            print(f"  -> Loaded {len(dataset)} samples from '{name}'.")

        except Exception as e:
            print(f"  [Error] Could not load or process dataset {name}: {e}")
            
    return pd.DataFrame(all_pairs)


def preprocess_text(text):
    if text is None:
        return ""
    return str(text).strip()


def get_last_layer_activations(texts, model, tokenizer, device):
    model.eval()
    vectors = []
    
    print(f"Extracting activation vectors for {len(texts)} samples...")
    for text in tqdm(texts, desc="Processing texts"):
        processed_text = preprocess_text(text)
        if not processed_text:
            processed_text = tokenizer.pad_token # Use pad token for empty inputs
            
        inputs = tokenizer(processed_text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, temperature=0.001)
        
        last_hidden_state = outputs.hidden_states[-1]
        sentence_vector = last_hidden_state.mean(dim=1).squeeze().cpu().float().numpy()
        vectors.append(sentence_vector)
        
    return np.array(vectors)


class ReconstructionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ReconstructionMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def train_model(mlp, train_loader, val_loader, epochs, lr, device):
    mlp.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(mlp.parameters(), lr=lr)
    
    print("\n--- Starting MLP Training ---")
    for epoch in range(epochs):
        mlp.train()
        train_loss = 0.0
        for anon_vecs, orig_vecs in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            anon_vecs, orig_vecs = anon_vecs.to(device), orig_vecs.to(device)
            
            optimizer.zero_grad()
            predicted_vecs = mlp(anon_vecs)
            loss = criterion(predicted_vecs, orig_vecs)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * anon_vecs.size(0)
            
        mlp.eval()
        val_loss = 0.0
        with torch.no_grad():
            for anon_vecs, orig_vecs in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                anon_vecs, orig_vecs = anon_vecs.to(device), orig_vecs.to(device)
                predicted_vecs = mlp(anon_vecs)
                loss = criterion(predicted_vecs, orig_vecs)
                val_loss += loss.item() * anon_vecs.size(0)
                
        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)
        
        print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
    print("--- Finished MLP Training ---")
    return mlp


def evaluate_model(mlp, test_loader, device):
    mlp.eval()
    mlp.to(device)
    all_predicted = []
    all_original = []
    
    print("\n--- Evaluating MLP on Test Set ---")
    with torch.no_grad():
        for anon_vecs, orig_vecs in tqdm(test_loader, desc="Evaluating"):
            anon_vecs = anon_vecs.to(device)
            predicted_vecs = mlp(anon_vecs)
            all_predicted.append(predicted_vecs.cpu().numpy())
            all_original.append(orig_vecs.cpu().numpy())
            
    all_predicted = np.concatenate(all_predicted, axis=0)
    all_original = np.concatenate(all_original, axis=0)
    
    mse = mean_squared_error(all_original, all_predicted)
    mae = mean_absolute_error(all_original, all_predicted)
    r2 = r2_score(all_original, all_predicted)
    
    cos_sim_list = [
        np.dot(orig, pred) / (np.linalg.norm(orig) * np.linalg.norm(pred))
        for orig, pred in zip(all_original, all_predicted)
    ]
    avg_cos_sim = np.mean(cos_sim_list)

    pearson_corr_list = [
        pearsonr(orig, pred)[0] 
        for orig, pred in zip(all_original, all_predicted)
    ]
    avg_pearson_corr = np.mean(pearson_corr_list)
    
    print("\n--- Evaluation Results ---")
    print(f"Mean Squared Error (MSE):      {mse:.6f}")
    print(f"Mean Absolute Error (MAE):     {mae:.6f}")
    print(f"R-squared (R²):                {r2:.6f}")
    print(f"Average Cosine Similarity:     {avg_cos_sim:.6f}")
    print(f"Average Pearson Correlation:   {avg_pearson_corr:.6f}")
    print("--------------------------\n")
    return mse, mae, r2, avg_cos_sim


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
    
    model_hidden_size = model.config.hidden_size
    print(f"Model hidden size: {model_hidden_size}")

    data_df = load_and_prepare_data(DATASETS_TO_LOAD, MAX_SAMPLES_PER_DATASET)
    
    if not data_df.empty:
        original_vectors = get_last_layer_activations(data_df["original"].tolist(), model, tokenizer, device)
        anonymized_vectors = get_last_layer_activations(data_df["anonymized"].tolist(), model, tokenizer, device)

        df_original_vectors = pd.DataFrame(original_vectors)
        df_anonymized_vectors = pd.DataFrame(anonymized_vectors)
        print("\nShape of original vectors dataframe:", df_original_vectors.shape)
        print("Shape of anonymized vectors dataframe:", df_anonymized_vectors.shape)

        X_train_val, X_test, y_train_val, y_test = train_test_split(
            anonymized_vectors, original_vectors, test_size=TEST_SET_SIZE, random_state=42
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=(VALIDATION_SET_SIZE / (1-TEST_SET_SIZE)), random_state=42
        )
        train_dataset = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float())
        val_dataset = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).float())
        test_dataset = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float())
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        mlp = ReconstructionMLP(input_dim=model_hidden_size, hidden_dim=model_hidden_size*2, output_dim=model_hidden_size)
        trained_mlp = train_model(mlp, train_loader, val_loader, EPOCHS, LEARNING_RATE, device)
        
        evaluate_model(trained_mlp, test_loader, device)
    else:
        print("No data was loaded. Exiting.")

import joblib 
joblib.dump(trained_mlp, mlp_path)

In [ ]:
# Forget space creation

class ReconstructionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ReconstructionMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def preprocess_text(text):
    if text is None:
        return ""
    return str(text).strip()

def get_last_layer_activations(texts, model, tokenizer, device):
    model.eval()
    vectors = []
    print(f"Extracting activation vectors for {len(texts)} samples...")
    for text in tqdm(texts, desc="Processing texts"):
        processed_text = preprocess_text(text)
        if not processed_text:
            processed_text = tokenizer.pad_token

        inputs = tokenizer(processed_text, return_tensors="pt", truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        last_hidden_state = outputs.hidden_states[-1]
        sentence_vector = last_hidden_state.mean(dim=1).squeeze().cpu().float().numpy()
        vectors.append(sentence_vector)

    return np.array(vectors)


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Loading trained MLP'...")
    try:
        mlp_model = joblib.load(mlp_path) 
        mlp_model.to(device)
        mlp_model.eval() 
        print("MLP model loaded successfully.")
    except FileNotFoundError:
        print(f"[Error] Model file not found'.")
        print("Please make sure you have saved the trained model")
        exit()
    except Exception as e:
        print(f"[Error] An error occurred while loading the MLP model: {e}")
        exit()
    
    print(f"Loading forget dataset...")
    questions = forget_dataset["anonymized_question"]
    input_vectors = get_last_layer_activations(questions, model, tokenizer, device)

    print("Passing activation vectors through the MLP for prediction...")
    input_tensor = torch.from_numpy(input_vectors).float().to(device)
    
    estimated_vectors_list = []
    with torch.no_grad():
        for i in tqdm(range(0, len(input_tensor), 32), desc="Predicting with MLP"):
            batch = input_tensor[i:i+32]
            predicted_batch = mlp_model(batch)
            estimated_vectors_list.append(predicted_batch.cpu().numpy())

    estimated_vectors = np.concatenate(estimated_vectors_list, axis=0)

    df_estimated_vectors = pd.DataFrame(estimated_vectors)
    df_estimated_vectors.index.name = "sample_index"

    print("\n--- Inference Complete ---")
    print("Successfully generated estimated activation vectors.")
    print(f"Shape of the final DataFrame: {df_estimated_vectors.shape}")
    print("\nFirst 5 estimated vectors:")
    print(df_estimated_vectors.head())
    
    output_filename = forget_space_path
    df_estimated_vectors.to_csv(output_filename)
    print(f"\nResults saved to '{output_filename}'")

In [ ]:
# Subspace creation


INPUT_VECTORS_FILE = forget_space_path
VARIANCE_TO_RETAIN = 0.95 


if __name__ == "__main__":
    print(f"Loading estimated vectors from '{INPUT_VECTORS_FILE}'...")
    if not os.path.exists(INPUT_VECTORS_FILE):
        print(f"[Error] Input file not found: '{INPUT_VECTORS_FILE}'")
        print("Please run the inference script first to generate this file.")
        exit()
        
    df_vectors = pd.read_csv(INPUT_VECTORS_FILE, index_col=0).T
    print(f"Successfully loaded data with shape: {df_vectors.shape}")

    X = df_vectors.values

    print("\nApplying Principal Component Analysis (PCA)...")
    pca = PCA()
    pca.fit(X)

    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    
    n_components_to_keep = np.argmax(cumulative_variance >= VARIANCE_TO_RETAIN) + 1
    
    print(f"Number of components to retain {VARIANCE_TO_RETAIN*100}% variance: {n_components_to_keep}")
    print(f"This reduces the dimensionality from {X.shape[1]} to {n_components_to_keep}.")

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='.', linestyle='--')
    plt.axhline(y=VARIANCE_TO_RETAIN, color='r', linestyle='-', label=f'{VARIANCE_TO_RETAIN*100}% Variance Threshold')
    plt.axvline(x=n_components_to_keep, color='g', linestyle='-', label=f'{n_components_to_keep} Components')
    plt.xlabel("Number of Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("PCA: Explained Variance vs. Number of Components")
    plt.grid(True)
    plt.legend()
    variance_plot_file = "pca_explained_variance.png"
    plt.savefig(variance_plot_file)
    print(f"Saved explained variance plot to '{variance_plot_file}'")
    plt.show()

    print(f"\nCreating final PCA model with {n_components_to_keep} components...")
    final_pca = PCA(n_components=n_components_to_keep)
    forget_subspace = final_pca.fit_transform(X)
    print(f"Shape of the final 'forget_subspace': {forget_subspace.shape}")
    np.save(OUTPUT_SUBSPACE_FILE, forget_subspace)
    print(f"Successfully saved the forget subspace to '{OUTPUT_SUBSPACE_FILE}'")

In [ ]:
# Unlearning filter
from scipy.stats import iqr

Ur = 2 * ((forget_subspace - np.min(forget_subspace)) / (np.max(forget_subspace) - np.min(forget_subspace))) - 1
print(f"Shape of the subspace basis U (n_features, n_components): {Ur.shape}")

projection_matrix_Ur = Ur @ Ur.T
projection_matrix_Ur = 2 * ((projection_matrix_Ur - np.min(projection_matrix_Ur)) / (np.max(projection_matrix_Ur) - np.min(projection_matrix_Ur))) - 1
print(f"Shape of the projection matrix UU^T: {projection_matrix_Ur.shape}")

identity_matrixr = np.identity(projection_matrix_Ur.shape[1])
print(f"Shape of the identity matrix I: {identity_matrixr.shape}")

unlearning_filterr = identity_matrixr - projection_matrix_Ur
print(f"Shape of the final unlearning filter: {unlearning_filterr.shape}")

unlearning_filter = 2 * ((unlearning_filterr - np.min(unlearning_filterr)) / (np.max(unlearning_filterr) - np.min(unlearning_filterr))) - 1
np.save(UNLEARNING_FILTER_PATH, unlearning_filter)

In [ ]:
# Saving unlearned model

unlearning_filter = np.load(UNLEARNING_FILTER_PATH)

def make_unlearning_permanent(model, unlearning_filter, layer_to_modify, alpha):
    device = model.device
    original_dtype = model.dtype 
    hidden_dim = model.config.hidden_size
    num_layers = len(model.model.layers)

    print("Constructing unlearning transformation matrix M in high precision...")
    P = torch.tensor(unlearning_filter, dtype=torch.float32, device=device)
    I = torch.eye(hidden_dim, dtype=torch.float32, device=device)
    M = I - alpha * (I-P)
    M_t = M.T

    if layer_to_modify == num_layers - 1:
        print(f"Applying unlearning filter to the final layer. Modifying lm_head.")
        target_layer = model.lm_head
        with torch.no_grad():
            original_weight_fp32 = target_layer.weight.data.float()
            new_weight_fp32 = original_weight_fp32 @ M_t
            target_layer.weight.data = new_weight_fp32.to(original_dtype)
        print("Unlearning permanently applied to the lm_head.")
        
    else:
        target_layer_idx = layer_to_modify + 1
        print(f"Applying unlearning filter to weights of layer {target_layer_idx} to influence KV cache...")
        
        target_layer = model.model.layers[target_layer_idx]
        layers_to_update = [
            target_layer.self_attn.q_proj,
            target_layer.self_attn.k_proj,
            target_layer.self_attn.v_proj
        ]

        with torch.no_grad():
            for layer in layers_to_update:
                original_weight_fp32 = layer.weight.data.float()
                new_weight_fp32 = original_weight_fp32 @ M_t
                layer.weight.data = new_weight_fp32.to(original_dtype)
        
        print(f"Unlearning permanently applied to attention weights of layer {target_layer_idx}.")
        
    return model

original_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="cuda:0",
        torch_dtype=torch.bfloat16 
    )

permanent_model = make_unlearning_permanent(original_model, unlearning_filter, LAYER_TO_MODIFY, UNLEARNING_STRENGTH_ALPHA)
permanent_model.save_pretrained(SAVE_DIRECTORY)